# 04 - Tabelas gerenciadas x não gerenciadas

Compara os dois tipos de tabela Delta no Spark:

| Tipo | Quem controla os dados | Como criamos aqui |
|---|---|---|
| **Não gerenciada (externa)** | O usuário (os dados ficam no MinIO) | `CREATE TABLE ... USING delta LOCATION 's3a://bronze/...'` |
| **Gerenciada (interna)** | O Spark (dados + metadados) | `df.write.format('delta').saveAsTable(...)` |

O teste decisivo é o `DROP TABLE`: o que acontece com os dados?

**Pré-requisito:** notebook `02` executado.

## 1. SparkSession

In [1]:
import os
from pathlib import Path
from dotenv import load_dotenv

RAIZ = Path.cwd().parent if Path.cwd().name == 'notebook' else Path.cwd()
if not (RAIZ / '.env').exists():
    raise FileNotFoundError('Arquivo .env não encontrado. Rode: cp .env.example .env')
load_dotenv(RAIZ / '.env', override=True)

from pyspark.sql import SparkSession
from delta import *
from delta.tables import DeltaTable

MINIO_ENDPOINT   = os.getenv('MINIO_ENDPOINT')
MINIO_ACCESS_KEY = os.getenv('MINIO_ACCESS_KEY')
MINIO_SECRET_KEY = os.getenv('MINIO_SECRET_KEY')
LANDING_BUCKET   = os.getenv('MINIO_LANDING_BUCKET')
BRONZE_BUCKET    = os.getenv('MINIO_BRONZE_BUCKET')

spark = (
    SparkSession.builder
    .appName('Gerenciadas_x_Nao_Gerenciadas')
    .master('local[*]')
    .config('spark.jars.packages', 'io.delta:delta-spark_2.12:3.2.1,org.apache.hadoop:hadoop-aws:3.3.4')
    .config('spark.sql.extensions', 'io.delta.sql.DeltaSparkSessionExtension')
    .config('spark.sql.catalog.spark_catalog', 'org.apache.spark.sql.delta.catalog.DeltaCatalog')
    .config('spark.sql.warehouse.dir', str(RAIZ / 'spark-warehouse'))
    # MinIO / S3A
    .config('spark.hadoop.fs.s3a.endpoint', MINIO_ENDPOINT)
    .config('spark.hadoop.fs.s3a.access.key', MINIO_ACCESS_KEY)
    .config('spark.hadoop.fs.s3a.secret.key', MINIO_SECRET_KEY)
    .config('spark.hadoop.fs.s3a.path.style.access', 'true')
    .config('spark.hadoop.fs.s3a.impl', 'org.apache.hadoop.fs.s3a.S3AFileSystem')
    .config('spark.hadoop.fs.s3a.connection.ssl.enabled', 'false')
    .getOrCreate()
)
print('SparkSession criada! Spark', spark.version)

import shutil
WAREHOUSE = RAIZ / 'spark-warehouse'
print('Warehouse (onde ficam as tabelas gerenciadas):', WAREHOUSE)

26/10/12 19:16:53 WARN Utils: Your hostname, DESKTOP-73OKR5D resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/12 19:16:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/thomasmaggi/pesquisadados_parte2/.venv/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/thomasmaggi/.ivy2/cache
The jars for the packages stored in: /home/thomasmaggi/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-81747f06-6e0c-4af5-988d-1bbb399c294b;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 315ms :: artifacts dl 16ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	io.delta#delta-spark_2.12;3.2.1 from central in [default]
	io.delta#delta-storage;3.2.1 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	org.a

SparkSession criada! Spark 3.5.3
Warehouse (onde ficam as tabelas gerenciadas): /home/thomasmaggi/pesquisadados_parte2/spark-warehouse


## 2. Tabela NÃO gerenciada (externa)

Criada com `LOCATION`, apontando para os dados que já estão no bucket `bronze`.

In [2]:
spark.sql("DROP TABLE IF EXISTS clientes_externa")
spark.sql(f"""
    CREATE TABLE clientes_externa
    USING delta
    LOCATION 's3a://{BRONZE_BUCKET}/clientes'
""")

spark.sql('DESCRIBE EXTENDED clientes_externa') \
    .filter("col_name IN ('Type', 'Provider', 'Location')").show(truncate=False)

26/10/12 19:17:14 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


+--------+---------------------+-------+
|col_name|data_type            |comment|
+--------+---------------------+-------+
|Type    |EXTERNAL             |       |
|Location|s3a://bronze/clientes|       |
|Provider|delta                |       |
+--------+---------------------+-------+



## 3. Tabela gerenciada (interna)

Criada a partir de um **DataFrame** (lido do `bronze`) com `saveAsTable`. Sem `LOCATION`,
o Spark decide onde guardar os dados (dentro do warehouse).

In [3]:
shutil.rmtree(WAREHOUSE / 'clientes_gerenciada', ignore_errors=True)
spark.sql("DROP TABLE IF EXISTS clientes_gerenciada")

df = spark.read.format('delta').load(f's3a://{BRONZE_BUCKET}/clientes')
df.write.format('delta').saveAsTable('clientes_gerenciada')

spark.sql('DESCRIBE EXTENDED clientes_gerenciada') \
    .filter("col_name IN ('Type', 'Provider', 'Location')").show(truncate=False)
spark.sql('SELECT * FROM clientes_gerenciada ORDER BY id_cliente').show()

26/10/12 19:17:20 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+--------+-------------------------------------------------------------------------------+-------+
|col_name|data_type                                                                      |comment|
+--------+-------------------------------------------------------------------------------+-------+
|Type    |MANAGED                                                                        |       |
|Location|file:/home/thomasmaggi/pesquisadados_parte2/spark-warehouse/clientes_gerenciada|       |
|Provider|delta                                                                          |       |
+--------+-------------------------------------------------------------------------------+-------+



+----------+----------+-------------+---+
|id_cliente|      nome|       cidade| uf|
+----------+----------+-------------+---+
|         1| Ana Souza|     Criciúma| SC|
|         2|Bruno Lima|Florianópolis| SC|
|         3|Carla Dias| Porto Alegre| RS|
|         4|Diego Melo|     Curitiba| PR|
+----------+----------+-------------+---+



## 4. DROP TABLE — o que acontece com os dados?

### 4.1 Não gerenciada: o DROP remove só os metadados

In [4]:
spark.sql('DROP TABLE clientes_externa')

caminho = f's3a://{BRONZE_BUCKET}/clientes'
print('Os dados continuam no MinIO?', DeltaTable.isDeltaTable(spark, caminho))
spark.read.format('delta').load(caminho).orderBy('id_cliente').show()

Os dados continuam no MinIO? True
+----------+----------+-------------+---+
|id_cliente|      nome|       cidade| uf|
+----------+----------+-------------+---+
|         1| Ana Souza|     Criciúma| SC|
|         2|Bruno Lima|Florianópolis| SC|
|         3|Carla Dias| Porto Alegre| RS|
|         4|Diego Melo|     Curitiba| PR|
+----------+----------+-------------+---+



### 4.2 Gerenciada: o DROP remove metadados e dados

In [5]:
pasta = WAREHOUSE / 'clientes_gerenciada'
print('Pasta existe ANTES do DROP?', pasta.exists())

spark.sql('DROP TABLE clientes_gerenciada')

print('Pasta existe DEPOIS do DROP?', pasta.exists())

Pasta existe ANTES do DROP? True
Pasta existe DEPOIS do DROP? False


## 5. Comparação

| | Não gerenciada (externa) | Gerenciada (interna) |
|---|---|---|
| Criação | `USING delta LOCATION '...'` | `saveAsTable(...)` a partir de um DataFrame |
| Onde ficam os dados | Onde o usuário definir (aqui, MinIO `bronze`) | No warehouse do Spark |
| `DESCRIBE EXTENDED` mostra | `Type: EXTERNAL` | `Type: MANAGED` |
| `DROP TABLE` | Remove só os metadados; **dados preservados** | Remove metadados **e dados** |
| Quando usar | Dados compartilhados ou vindos de outro sistema (lakehouse) | Tabelas temporárias ou de uso interno do Spark |

**Nos dois laboratórios:**
- **Trabalho 1:** as tabelas Delta foram criadas com `LOCATION` (não gerenciadas); as tabelas Iceberg ficaram no catálogo `local`, e nele usamos `DROP TABLE ... PURGE` para remover também os dados.
- **Trabalho 2:** as tabelas do `bronze` são não gerenciadas, e a tabela criada por DataFrame é gerenciada.

In [6]:
spark.stop()
print('SparkSession finalizada.')

SparkSession finalizada.
